In [1]:
# 基础组件Model

## 模型微调代码实例 
### 任务类型
#### 文本分类

## 使用模型
### hfl/rbt3

## 数据集地址


In [5]:
# 文本分类

## step1 导包
from transformers import AutoTokenizer,AutoModelForSequenceClassification

In [6]:
# 加载数据
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data.head()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [7]:
# 删除空数据
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [8]:
## 创建DataSet
from torch.utils.data import Dataset

class MyDataSet(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"],  self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)
        

In [9]:
dataset = MyDataSet()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', 1)
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', 1)
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', 1)
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', 1)
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', 1)


In [43]:
# 划分数据集
from torch.utils.data import random_split

train_len = int(0.9 * len(dataset))
valid_len = len(dataset) - train_len
trainset, validset = random_split(dataset, lengths=[train_len, valid_len])
len(trainset), len(validset)

(6988, 777)

In [13]:
for i in range(10):
    print(trainset[i])

('入住的是260元的迷你标准间。感觉比想象的要好很多，房间如果住一个人很合适的，洗手间很大，很干净。房间里床上用品质量很不错，整洁如新。房间窗户为玻璃幕墙，无法打开换气。这是距离大连火车站最近的酒店。前台服务态度很好，很热情，业务熟练。就是退房的时候，查房比较慢，人多的时候，要多等几分钟。总体感觉还是可以的，交通极为便利。', 1)
('由于到店时间为凌晨1:00多，因此,估计房间比我预定的要升一个档次，很宽敞。而且早上可以看到外面的景色及鸟的叫声。不过早上是被鸟叫的声音吵醒的……室内有保险柜，感觉没太大用处；有冰箱，比较实用……早餐不好吃，不知道是多少钱的标准……', 1)
('刚住了两个美丽华240的贵宾楼感觉有点贵160比较合适吃早餐最好早点去晚了就没了一层的“碘泉”还不错可以去泡泡据德州本地人说那的girl是当', 1)
('大约2年前住过一次,也许是新装修过的原因,这次去感觉比上次要好,房间感觉不错,服务也蛮好,酒店租车的价钱也算公道', 1)
('房间还算可以，但硬件设施偏差，服务水平也很一般。', 1)
('很有品位的酒店,服务也非常到位.尤其感慨的是:能把酒店的客房布置成乡村旅舍的格调,既淳朴,又不失现代生活的方便,这个要赞一个!另外,酒店贴心的服务也让人感到温馨。入住酒店时，已经过8点，还没有就餐，酒店有位男性服务员（不知道是否是领班）脸上只出现了少许的犹豫，马上答应马上安排，让我们一家人真的很感动。菜品也很有特色。离开酒店的时候，想不到还有细致服务的感受，停车场服务员一见我发动汽车，马上过来递上免费的停车票，告知待会出去交给外面的收费员。这种细致的服务，确实体现了这个酒店的管理理念。提一个建议，从环城高速进入酒店的指示牌能否换成醒目点的，晚上入住酒店的时候，为找路花了不少的时间。', 1)
('酒店周末都系学生情侣入住，噪音实在恐怖，发现很多学生在前台的价格系120元哦.环境，设施实在系太差了!不建议住', 0)
('性价比还是不错的,房间也很干净,周围也比较安静,唯一的缺点就是房间有点小,我住的大床房间', 1)
('网上讲的房间是24平米，实际没有那么大。另外房的空调也不是很凉快!', 1)
('酒店周边环境尚可,离火车站5公里左右,算远,房间空气不行,较旧.不推荐.', 0)


In [48]:
tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")
import torch

## 定义打包函数
def collate_func(batch):
    texts, labels = [],[]
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts,max_length=128,padding="max_length",truncation=True, return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs


In [49]:
# 创建DataLoader
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset,batch_size=32,shuffle=True,collate_fn=collate_func) #shuffle为打乱
validloader = DataLoader(validset,batch_size=64,shuffle=False,collate_fn=collate_func)


## DataSet 只返回一条 DataLoader 返回一个batch

In [38]:
# trainloader[i] 是错误的
## 读取数据
next(enumerate(trainloader))

['第一次去郑州就入住的宾馆.去的时候打表被出租车司机黑了,后来知道了,到郑州打车可以讲价的,机场到中州皇冠90元足够了,如果是坎价高手估计80也能下来吧.有免费的游泳池很不错,但游泳池不太好找,而且去的路上要小心台阶,本人就不小心拌倒了,因为没有明示,这点酒店做的不到位.', '环境比较旧，但是床单、毛巾都比较干净，前台服务态度良好。', '交通很方便，离建国门地铁站步行5分钟，但是房间太小，感觉很老气的房子，进屋后觉得很压抑。房间内味道也不是很好。个人认为性价比差。', 'GOOD不错的酒店,还是好呀OK不错的', '总体来说不错。交通比较方便，早餐也可以。步行去沃尔玛购物约十分钟。去旅游的话还是开车方便，景点可以去多去几处。补充点评2008年4月13日：小心电梯夹人，合上的速度太快了，很少见。', '酒店的环境,房间和服务都不错,但是门口的保安就很差.让人联想到"打手".再加上网上也有对保安的说法,以前还不以为然,现在不得不同意他们的说法了.宾馆反馈2008年6月12日：尊敬的客人：您好！看完您的点评，酒店曾第一时间试图联系您了解事情的始末，但未能联系到，我们深感遗憾！同时，酒店立即以书面形式责成酒店外保所属的物业公司需加强对安保的培训和督导，并以该案例为戒，切实做好对客服务工作，避免类似事件的再次发生，否则酒店有权中止和物业的合作。如家始终秉承为每一位宾客提供安全、洁净、温馨的住店环境，也真诚的希望每一位住店客人能对酒店的工作进行督导和建议。我们感谢所有帮助酒店健康成长的客人！', '接站免费司机是四川人很实在,在路上告诉了一些初到拉萨的注意事项,宾馆很小,不是我们内地的星级标准.早餐还行,卫生间很大,就是价格稍高,我们到的第二天停电,所以我们换宾馆了.', '我订的是商务大床房，给我的房间是8418房，床是挺大的，但是房间太小了，基本上是进屋就上床了，连椅子都放不开，但是有个看不到海景的大阳台，不知道是怎么设计的，估计这房应该是个单人房改的。如家的电视节目有马赛克，给前台打电话，他们说叫工程师来看看，结果过了几分钟来了个拿对讲机的大婶，告诉我们是信号的原因，所有房间的节目都这样，要到晚上才能好，我们想那就算了，可是过会和我们同行来的另一个房间的朋友说他们的电视信号一直很好，气死了，如家竟然这样骗顾客，给前台打电话，又改口说是部分房间的信号是这样。晚上8点多

(0,
 {'input_ids': tensor([[ 101, 5018,  671,  ..., 4157, 6983,  102],
         [ 101, 4384, 1862,  ...,    0,    0,    0],
         [ 101,  769, 6858,  ...,    0,    0,    0],
         ...,
         [ 101, 1384, 4917,  ...,    0,    0,    0],
         [ 101, 8108, 3299,  ..., 3341,  117,  102],
         [ 101, 1057,  857,  ..., 2361, 3307,  102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 0, 0, 0])})

In [39]:
# 创建模型以及优化器
from torch.optim import Adam
model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

optimizer = Adam(model.parameters(), lr=2e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
# 训练与验证

def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            output = model(**batch)
            pred = torch.argmax(output.logits,dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)


def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep},global: {global_step},loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep},acc: {acc}")

In [46]:
## 模型训练
train()

['如果用一句话来点评：服务远远好过硬件。这次入住了行政湖景房和豪华商务房，感受比较多，就多说几句。入住当天虽然没有如约拿到双床房（可能入住人太多，说是还没有打扫好），但是前台安排了最高层的湖景房，补偿了一下。很少见的是每间房只提供单早，每次都要签单，稍微有点麻烦。不过早餐45一份，内容也挺丰富，还有小笼蒸的糕点，很喜欢。在酒店中餐厅消费可以打85折，价格很适中，即使人少也能提供包间，还不设最低消费，不错。离西湖确实比较近，慢慢散步十五分钟左右绝对可以达到。在湖景房或者17楼湖景餐厅还可以看到西湖喷泉、白堤，很享受。门童非常积极主动，除了热情招呼，还会主动询问是否需要拦出租车，知道我们要去机场，还告诉我们等车的位置以及可以在酒店大堂购买机场大巴的车票。客房服务很及时，需要什么东西都会尽快送到，打扫卫生迅速到令人吃惊，我们才一顿早餐的工夫，回来屋子已经收拾整齐。前台工作人员让我印象最深刻的是一名叫欣怡的姑娘，耐心、亲切，在我提出要定多一间房，想看看房后，她很快处理完自己手中的事陪我们前往，提供了很多有帮助的信息说到房间，这确实是唯一令我觉得与这间酒店很不太相称的地方。虽然我很欣赏客房设施的全面，免费提供的水果和饮料，卫生间里既有浴缸又有淋浴房的多种选择。但是，不得不说，设施实在是“太”陈旧了，旧的衣柜的门开闭都有吱吱嘎嘎的声响，还要往上提一提才能关严实。旧的马桶里的水阀的链子都是断的用绳子修补好的，结果还是一个劲儿的断，在我们自己维修过两次后，不得已求助客房服务，他们立刻通知工程部的维修人员，很快换上了新的带铁链的，接着即刻又有客房服务人员打扫干净，衔接的那么好，服务总是无可挑剔。中途加定的豪华客房据说是按照五星级标准装修的，不过装修的品位着实不怎么样，所以也不显得新到哪里去。（而且后来通过携程预定到的房比之前我们去看的房小了不少，也没有之前看到的组合沙发一块儿，不知道是不是因为通过携程预定会比直接在酒店预定少那么二三十块钱就缩水的缘故，想不明白，同是5-7层的豪华房，怎么大小，设备还有区别呢？）建议向皇冠假日酒店或者新开元酒店的设计风格学习学习。如果酒店能够重新装修一下，我想即使价格再高一点，我还是会选择马可波罗。宾馆反馈2008年8月9日：非常感谢您我们酒店中肯的点评。您加订的是商务豪华双床房，而您参观的是商务豪华大床房。由于房间格局的不同，商务豪华双床房内由带琴

KeyboardInterrupt: 

In [50]:
# 轻量验证：单步前向 + 一次评估
batch = next(iter(trainloader))
output = model(**batch)
print("loss:", float(output.loss.detach()))
print("eval acc:", float(evaluate()))

loss: 0.40068623423576355
eval acc: 0.8301158547401428


In [59]:
# Step9 模型预测
id2_label = {0:"差评!",1:"好评!"}
sen = "我觉得这家店很好吃"
with torch.inference_mode():
    inputs=tokenizer(sen, return_tensors="pt")
    inputs = {k:v for k,v in inputs.items()}
    model = model.to("cpu")
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入我觉得这家店很好吃
模型预测结果:好评!
